# 02 — A forest under wind

Companion to [`examples/grow_a_forest.py`](../examples/grow_a_forest.py).
We seed a disk-shaped plot with ~20 saplings, run the simulation for 80
generations and watch the population thin itself out under wind and
competition for light.

## Why this matters

Foresters know empirically that as a stand matures, the total number of
stems falls while the average stem biomass rises — and the two are
linked by a remarkably regular power law,
$\bar{M} \propto N^{-3/2}$ (Yoda's self-thinning rule). MechaTree
produces this regularity from first principles: trees that grow tall
fast intercept more light, shade their neighbours, then over-reach and
snap in a storm. No global rule, just mechanics + light.

Eloy et al. 2017, fig. 4, plots exactly this curve.

In [ ]:
from dataclasses import replace
from pathlib import Path

import plotly.graph_objects as go

import mechatree as mt

mt.figstyle.apply()

cfg = mt.load_config(Path("../examples/forest.yaml").resolve())
# Shrink the disk so the notebook runs in seconds rather than minutes.
cfg = replace(
    cfg,
    forest=replace(cfg.forest, n_trees_init=20, n_trees_max=300, size=30.0),
)
cfg.forest

## Run the simulation

`Forest.run` drives the per-generation lifecycle: light competition
across the union of all leaves, mechanics + growth for each tree,
pruning, death (low branch count after a grace period, or old age),
birth (random placement until `n_trees_max`).

In [ ]:
n_generations = 80
history = []  # (gen, n_trees, biomass, n_branches)

forest = mt.Forest(cfg, seed=42)


def on_step(gen, _forest, stats):
    history.append((gen, stats.n_trees, stats.biomass_total, stats.n_branches_total))
    if gen % 10 == 0:
        print(
            f"gen={gen:>4}: trees={stats.n_trees:>4}  biomass={stats.biomass_total:>8.2f}  "
            f"+born={stats.n_born:<3} -died={stats.n_died:<3}"
        )


forest.run(n_generations, on_step=on_step)
print(f"\nFinal: {history[-1][1]} trees, biomass={history[-1][2]:.2f}")

## Population & biomass over time

Two curves on a twin-axis chart: total stems alive (green, left axis),
total biomass (brown, right axis). Stems drop while biomass climbs —
the survivors get bigger faster than the deaths take away.

In [ ]:
gens = [h[0] for h in history]
trees_over_time = [h[1] for h in history]
biomass_over_time = [h[2] for h in history]

fig = mt.figstyle.figure(size="full", aspect=9 / 5)
fig.add_trace(
    go.Scatter(
        x=gens,
        y=trees_over_time,
        name="trees alive",
        line=dict(color=mt.figstyle.COLORS["green"]),
    )
)
fig.add_trace(
    go.Scatter(
        x=gens,
        y=biomass_over_time,
        name="biomass",
        line=dict(color=mt.figstyle.COLORS["brown"]),
        yaxis="y2",
    )
)
fig.update_layout(
    title="Population & biomass over time",
    xaxis_title="generation",
    yaxis=dict(
        title=dict(text="trees alive", font=dict(color=mt.figstyle.COLORS["green"])),
        tickfont=dict(color=mt.figstyle.COLORS["green"]),
    ),
    yaxis2=dict(
        title=dict(text="total biomass", font=dict(color=mt.figstyle.COLORS["brown"])),
        tickfont=dict(color=mt.figstyle.COLORS["brown"]),
        overlaying="y",
        side="right",
    ),
)
fig.show()

## Where the survivors stand

`plot_forest_topdown` projects the final stand onto the ground plane.
Each disk is sized by trunk biomass; the outer circle is the disk
boundary.

In [ ]:
fig = mt.plot_forest_topdown(forest)
fig.update_layout(title=f"Final stand (n={len(forest.trees)})")
fig.show()

## Self-thinning, log-log

`plot_self_thinning` gives the canonical three-panel diagnostic: $N(t)$,
$M(t)$ and $\bar{M}$ vs $N$ on log axes. The dashed reference is the
Yoda −3/2 slope. The runs in this notebook are small enough that the
trajectory is short; expand `cfg.forest.size` and `n_generations` to see
the asymptote more clearly.

In [ ]:
fig = mt.plot_self_thinning([(g, n, m) for g, n, m, _ in history])
fig.show()

## Where to next

- [03_neural_genome.ipynb](03_neural_genome.ipynb) — single-tree
  comparison of the default constant genome with two evolved genomes.
- [05_strahler_diagnostics.ipynb](05_strahler_diagnostics.ipynb) — the
  self-similar branching that emerges inside each surviving tree.
- User guide: [Simulating a forest](../docs/userguide.rst).